# FingerBall Stage 2J-1 — Multi-model 2D / 3D Hand Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2J1_MultiModel_2D3D_Comparison.ipynb)

### Goal
**Stage 2Hの目視確認済みXYを変更せず**、MediaPipe Tasks Hand LandmarkerとWiLoR-miniを同じ元動画フレームに推論させ、2D画素座標、推定3D、手の左右、モデル間の誤差を統一表で比較します。

- 元動画 1206×648 / frame 185–198、特に188–193（190はリリース目視参照、未自動検出）。
- 2D座標はすべて**元動画のピクセル座標**へ変換。モデルが異なる3D座標は**生値を別の座標系として記録**し、直接平均しません。
- 入力モデルの選択について**手動ROIを使用する条件（oracle-assisted）**と、モデル独立検出の条件を区別します。ROI assistedの2D誤差は**独立した検出性能の評価ではありません**。
- 遮蔽点は評価から除外。手動点、目視確認済み引継ぎ候補、AI点の出自を保持。
- 失敗した検出は欠損として出力し、成功したように補間しません。
- WiLoR-miniは内部で**MANO_RIGHT.pklをHugging Faceから自動取得**する実装です。MANOのモデルライセンスを必ず確認し、許諾を得るまで`ENABLE_WILOR=False`で実行してください。<https://mano.is.tue.mpg.de/>

参考：<https://github.com/warmshao/WiLoR-mini>、<https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python>

**これは比較研究用Notebookで、SOMAフィッティングは次のStage 2J-2です。**

## 0. Colabセットアップ：MediaPipe単独は常に実行可能

WiLoR-miniは旧Python 3.10想定で、最新Colabでは依存ライブラリとMANOライセンスの問題があり得ます。比較表はWiLoRが動かない場合も生成します。

In [ ]:
%pip -q install "mediapipe>=0.10.20" "opencv-python-headless>=4.8" "pandas>=2" "numpy>=1.24" "matplotlib>=3.7" "plotly>=5.18"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib,json,time,traceback,urllib.request,subprocess,sys
import numpy as np,pandas as pd,cv2,matplotlib.pyplot as plt
import mediapipe as mp
from IPython.display import display
print('Python',sys.version.split()[0],'MediaPipe',mp.__version__)

## 1. 設定・元データの読み込み

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
VIDEO_PATH=PROJECT/(VIDEO_STEM+'.mp4')  # 加工済みzoom動画は使わない
STAGE2H_CSV=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'hand_21_landmarks_2d_corrected.csv'
OUTPUT_DIR=PROJECT/'FingerBall_v2_stage2j1_results'/VIDEO_STEM
FRAME_START,FRAME_END=185,198
RELEASE_REFERENCE=190
THROWING_HAND='Left'
USE_MANUAL_ROI_FOR_MEDIAPIPE=True
ENABLE_WILOR=False   # MANOのライセンス・配布物の利用許諾を確認した場合のみ True
I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS=False
WILOR_RUN_DETECTOR_BASELINE=True
WILOR_RUN_MANUAL_BBOX_ASSIST=True
ROI_MARGIN=1.8
FRAME_DISPLAY=(188,189,190,191,192,193)
if THROWING_HAND not in ('Left','Right'):raise ValueError('Throwing hand must be Left/Right')
if ENABLE_WILOR and not I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS:
    raise PermissionError('WiLoR-miniはMANOモデルファイルを自動取得します。MANOの利用許諾を確認してからフラグを設定してください')
for file in (VIDEO_PATH,STAGE2H_CSV):
    if not file.is_file():raise FileNotFoundError(file)
if 'zoom' in VIDEO_PATH.stem.lower():raise ValueError('元動画を指定してください')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
source_hash=hashlib.sha256(STAGE2H_CSV.read_bytes()).hexdigest()
ref=pd.read_csv(STAGE2H_CSV)
req={'frame','landmark_id','x_px','y_px','annotation_status'}
if diff:=req-set(ref.columns):raise ValueError(f'Missing Stage2H columns: {diff}')
if ref.duplicated(['frame','landmark_id']).any():raise ValueError('Duplicate rows')
frames=list(range(FRAME_START,FRAME_END+1))
ref=ref[ref.frame.isin(frames)].copy()
assert len(ref)==21*len(frames)
assert all(ref.groupby('frame').landmark_id.nunique().eq(21))
cap=cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():raise RuntimeError('Video failed to open')
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
assert FRAME_END<int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert W>0 and H>0
def frame_rgb(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
    ok,bgr=cap.read()
    if not ok:raise RuntimeError(f'Could not read frame {f}')
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
images={f:frame_rgb(f) for f in frames}
cap.release()
reference={};status={}
for f in frames:
    a=ref[ref.frame==f].set_index('landmark_id').reindex(range(21))
    reference[f]=a[['x_px','y_px']].to_numpy(float)
    status[f]=a.annotation_status.astype(str).to_list()
print('Video',W,H,'FPS',FPS,'Frozen SHA256',source_hash)
print('Annotations',ref.annotation_status.value_counts().to_dict())

## 2. ROIと共通フォーマット

手動点に基づくROIは研究上**手動補助条件**です。独立した自動検出性能の評価と区別します。

In [ ]:
EDGE_SETS={'Thumb':[0,1,2,3,4],'Index':[0,5,6,7,8],
           'Middle':[0,9,10,11,12],'Ring':[0,13,14,15,16],
           'Little':[0,17,18,19,20]}
COLORS={'Thumb':'#d97706','Index':'#0891b2','Middle':'#be185d',
        'Ring':'#94a3b8','Little':'#64748b'}
NAME=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
      'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
      'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
      'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
      'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
def manual_box(f,margin=ROI_MARGIN):
    p=reference[f]
    p=p[np.isfinite(p).all(axis=1)]
    if len(p)<3:raise ValueError(f'Frame {f}: too few reference points for assisted ROI')
    lo=np.min(p,axis=0);hi=np.max(p,axis=0)
    center=(lo+hi)/2
    side=max(float(max(hi-lo))*margin,100.)
    x1=max(0,center[0]-side/2);y1=max(0,center[1]-side/2)
    x2=min(W,center[0]+side/2);y2=min(H,center[1]+side/2)
    if x2<=x1 or y2<=y1:raise ValueError('Invalid box')
    return np.array([x1,y1,x2,y2],float)
def image_crop(image,box,size=512):
    x1,y1,x2,y2=box
    side=max(x2-x1,y2-y1);cx=(x1+x2)/2;cy=(y1+y2)/2
    scale=size/side
    M=np.float32([[scale,0,size/2-cx*scale],[0,scale,size/2-cy*scale]])
    patch=cv2.warpAffine(image,M,(size,size),flags=cv2.INTER_LINEAR,
               borderMode=cv2.BORDER_CONSTANT,borderValue=(105,105,105))
    Minv=cv2.invertAffineTransform(M)
    return patch,Minv
def map_to_original(p,Minv):
    return np.concatenate([p,np.ones((len(p),1))],axis=1)@Minv.T
empty21=lambda:np.full((21,3),np.nan,float)
predictions={}
def store(f,method,xy=None,world=None,relative=None,handedness='',confidence=np.nan,
          detected=False,reason='',assisted=False):
    if xy is None:xy=np.full((21,2),np.nan)
    if world is None:world=empty21()
    if relative is None:relative=np.full((21,),np.nan)
    xy=np.asarray(xy,dtype=float).reshape(21,2)
    world=np.asarray(world,dtype=float).reshape(21,3)
    relative=np.asarray(relative,dtype=float).reshape(21)
    predictions[(int(f),method)]=dict(xy=xy,world=world,relative=relative,
         handedness=str(handedness),confidence=float(confidence),
         detected=bool(detected),reason=str(reason),assisted=bool(assisted))
def choose_hand(candidates,refxy,requested='Left'):
    # candidate dict: xy + hand label, score
    if not candidates:return None
    ranked=[]
    for c in candidates:
        good=np.isfinite(refxy).all(axis=1)&np.isfinite(c['xy']).all(axis=1)
        dist=float(np.median(np.linalg.norm(refxy[good]-c['xy'][good],axis=1))) if np.any(good) else 1e7
        handedness_penalty=0 if c.get('hand','')==requested else 200
        ranked.append((dist+handedness_penalty,c))
    return min(ranked,key=lambda v:v[0])[1]

## 3. MediaPipe Tasks：ROI補助と独立全画面検出の両方

画像座標に変換した21点、相対Z、model-estimated world 3Dを保存します。World 3Dは物理的なground truthではありません。

In [ ]:
task_path=PROJECT/'models'/'hand_landmarker.task'
task_path.parent.mkdir(parents=True,exist_ok=True)
if not task_path.is_file():
    url='https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    urllib.request.urlretrieve(url,task_path)
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
mp_opts=mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task_path)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,min_hand_detection_confidence=.15,min_hand_presence_confidence=.15)
def infer_mp(img,offset_transform=None):
    mp_image=mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(img))
    result=landmarker.detect(mp_image)
    candidates=[]
    for k,landmarks in enumerate(result.hand_landmarks):
        cxy=np.array([[p.x*img.shape[1],p.y*img.shape[0]] for p in landmarks])
        xy=map_to_original(cxy,offset_transform) if offset_transform is not None else cxy
        z=np.array([p.z for p in landmarks])
        world=np.array([[p.x,p.y,p.z] for p in result.hand_world_landmarks[k]]) if k<len(result.hand_world_landmarks) else empty21()
        h=result.handedness[k][0] if k<len(result.handedness) and result.handedness[k] else None
        candidates.append({'xy':xy,'relative':z,'world':world,'hand':h.category_name if h else '',
                           'score':float(h.score) if h else np.nan})
    return candidates
with mp_vision.HandLandmarker.create_from_options(mp_opts) as landmarker:
    for f in frames:
        # Independent full-frame baseline: no manual ROI.
        try:
            candidates=infer_mp(images[f])
            chosen=choose_hand(candidates,reference[f],THROWING_HAND)
            if chosen is None:store(f,'mediapipe_full',reason='no_hand')
            else:store(f,'mediapipe_full',xy=chosen['xy'],world=chosen['world'],
                   relative=chosen['relative'],handedness=chosen['hand'],
                   confidence=chosen['score'],detected=True)
        except Exception as e:
            store(f,'mediapipe_full',reason=f'{type(e).__name__}:{e}')
        if USE_MANUAL_ROI_FOR_MEDIAPIPE:
            try:
                patch,inv=image_crop(images[f],manual_box(f))
                chosen=choose_hand(infer_mp(patch,inv),reference[f],THROWING_HAND)
                if chosen is None:store(f,'mediapipe_manual_roi',reason='no_hand',assisted=True)
                else:store(f,'mediapipe_manual_roi',xy=chosen['xy'],world=chosen['world'],
                      relative=chosen['relative'],handedness=chosen['hand'],
                      confidence=chosen['score'],detected=True,assisted=True)
            except Exception as e:
                store(f,'mediapipe_manual_roi',reason=f'{type(e).__name__}:{e}',assisted=True)
print('MediaPipe results:',{name:sum(predictions[(f,name)]['detected'] for f in frames)
    for name in ('mediapipe_full','mediapipe_manual_roi') if (frames[0],name) in predictions})

## 4. WiLoR-mini（任意・GPU推奨）

**ライセンス確認後のみ有効化します。** WiLoR-mini実装は`MANO_RIGHT.pkl`を自動取得します。独立YOLO検出条件と、手動21点から作ったbounding boxを与える**oracle-assisted条件**を分けます。

WiLoR-miniの実装元はPython 3.10推奨です。現行Colabで依存関係が合わない場合は明確なエラーを残してスキップできます。モデルの配布可否は独立に確認してください。 <https://github.com/warmshao/WiLoR-mini>

In [ ]:
if ENABLE_WILOR:
    if not I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS:
        raise PermissionError('MANO model assets usage permission is required')
    import torch
    print('WiLoR-mini setup: Python',sys.version.split()[0])
    # Installation may fail in newer Colab Python. Fail explicitly, not silently.
    try:
        from wilor_mini.pipelines.wilor_hand_pose3d_estimation_pipeline import WiLorHandPose3dEstimationPipeline
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install',
          'git+https://github.com/warmshao/WiLoR-mini'],check=True)
        from wilor_mini.pipelines.wilor_hand_pose3d_estimation_pipeline import WiLorHandPose3dEstimationPipeline
    dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    wilor=WiLorHandPose3dEstimationPipeline(device=dev,
       dtype=torch.float16 if dev.type=='cuda' else torch.float32,verbose=False)
    def wilor_candidate(o,transform=None):
        if 'wilor_preds' not in o:return None
        pred=o['wilor_preds']
        xy=np.asarray(pred['pred_keypoints_2d'])[0,:,:2].astype(float)
        xyz=np.asarray(pred['pred_keypoints_3d'])[0,:,:3].astype(float)
        if xy.shape!=(21,2) or xyz.shape!=(21,3):
            raise ValueError(f'WiLoR returned unexpected 21 joints: {xy.shape}, {xyz.shape}')
        if transform is not None:xy=map_to_original(xy,transform)
        # WiLoR MANO wrapper already maps joints to OpenPose/MediaPipe-equivalent order.
        # For left hands, pred_keypoints_3d X has been mirrored in pipeline.
        return {'xy':xy,'world':xyz,'hand':'Right' if float(o['is_right'])>=.5 else 'Left',
                'score':np.nan}
    for f in frames:
        image=images[f]
        if WILOR_RUN_DETECTOR_BASELINE:
            try:
                outs=wilor.predict(image)
                candidates=[c for o in outs if (c:=wilor_candidate(o)) is not None]
                chosen=choose_hand(candidates,reference[f],THROWING_HAND)
                if chosen is None:store(f,'wilor_full_detector',reason='no_hand')
                else:store(f,'wilor_full_detector',xy=chosen['xy'],world=chosen['world'],
                           handedness=chosen['hand'],detected=True)
            except Exception as e:
                store(f,'wilor_full_detector',reason=f'{type(e).__name__}:{e}')
        if WILOR_RUN_MANUAL_BBOX_ASSIST:
            try:
                # Predict on original full image, supplying hand box in original pixels.
                bbox=manual_box(f).reshape(1,4).astype(np.float32)
                is_right=np.array([1 if THROWING_HAND=='Right' else 0])
                outs=wilor.predict_with_bboxes(image,bbox,is_right)
                chosen=wilor_candidate(outs[0]) if outs else None
                if chosen is None:store(f,'wilor_manual_bbox',reason='no_hand',assisted=True)
                else:store(f,'wilor_manual_bbox',xy=chosen['xy'],world=chosen['world'],
                           handedness=chosen['hand'],detected=True,assisted=True)
            except Exception as e:
                store(f,'wilor_manual_bbox',reason=f'{type(e).__name__}:{e}',assisted=True)
    del wilor
else:
    print('WiLoR-mini disabled. MediaPipe comparisons proceed normally.')
    for f in frames:
        for label,assisted in [('wilor_full_detector',False),('wilor_manual_bbox',True)]:
            store(f,label,reason='disabled_or_license_not_confirmed',assisted=assisted)
print('WiLoR detection:',{method:sum(predictions[(f,method)]['detected'] for f in frames)
      for method in ('wilor_full_detector','wilor_manual_bbox')})

## 5. 共通21点CSVと比較指標

長形式：**1 frame×21 landmarks×method**、手動補正ステータスとXY、各予測XY、相対Z/推定3D（モデルごとの元座標・単位）、エラー。未検出は全21点をNaNにして行を維持します。

In [ ]:
methods=['stage2h_reference','mediapipe_full','mediapipe_manual_roi',
         'wilor_full_detector','wilor_manual_bbox']
rows=[]
for f in frames:
    for method in methods:
        r=predictions.get((f,method))
        if method=='stage2h_reference':
            # No 3D ground truth; only user-reviewed 2D.
            r=dict(xy=reference[f],world=empty21(),relative=np.full(21,np.nan),
                   handedness=THROWING_HAND,confidence=np.nan,detected=True,
                   assisted=False,reason='manual_reviewed_xy_only')
        if r is None:
            r=dict(xy=np.full((21,2),np.nan),world=empty21(),
                   relative=np.full(21,np.nan),handedness='',confidence=np.nan,
                   detected=False,assisted=False,reason='not_run')
        for i in range(21):
            p=r['xy'][i]; gt=reference[f][i]; q=r['world'][i]
            valid=np.isfinite(gt).all() and status[f][i] not in ('occluded','missing')
            err=float(np.linalg.norm(p-gt)) if valid and np.isfinite(p).all() and method!='stage2h_reference' else np.nan
            rows.append({'frame':f,'time_s':f/FPS,'method':method,'landmark_id':i,
              'name':NAME[i],'status_stage2h':status[f][i],
              'stage2h_x_px':gt[0] if valid else np.nan,
              'stage2h_y_px':gt[1] if valid else np.nan,
              'x_px':p[0],'y_px':p[1],
              'z_relative':r['relative'][i],
              'model3d_x':q[0],'model3d_y':q[1],'model3d_z':q[2],
              'model3d_coord_system':('mediapipe_model_world' if method.startswith('mediapipe_') else
                    'wilor_mano_pred_keypoints_3d' if method.startswith('wilor_') else 'none'),
              'detected':bool(r['detected']),'assisted_by_stage2h_roi':bool(r['assisted']),
              'handedness_reported':r['handedness'],'score_reported':r['confidence'],
              'error_to_stage2h_px':err,'failure_reason':r['reason']})
comparison=pd.DataFrame(rows)
common_csv=OUTPUT_DIR/'stage2j1_common_21_landmarks_with_provenance.csv'
comparison.to_csv(common_csv,index=False)
summary=[]
for method,g in comparison[comparison.method!='stage2h_reference'].groupby('method'):
    values=g.error_to_stage2h_px.dropna()
    vals_manual=g.loc[g.status_stage2h=='manual_visible','error_to_stage2h_px'].dropna()
    summary.append({'method':method,
        'detected_frames':int(g.groupby('frame').detected.first().sum()),
        'total_frames':len(frames),
        'n_points_evaluable':len(values),
        'RMSE_px':float(np.sqrt(np.mean(values**2))) if len(values) else np.nan,
        'median_px':float(values.median()) if len(values) else np.nan,
        'manual_only_n':len(vals_manual),
        'manual_only_RMSE_px':float(np.sqrt(np.mean(vals_manual**2))) if len(vals_manual) else np.nan,
        'assisted_with_reference_ROI':bool(g.assisted_by_stage2h_roi.any())})
summary=pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR/'stage2j1_model_comparison_summary.csv',index=False)
display(summary)
print('CSV:',common_csv)